In [ ]:
import json
import time
from functools import lru_cache
from pathlib import Path
from typing import Any

import marimo as mo
import requests

DELAY = 0.3

POKE_ID_MAX = 151


@lru_cache(maxsize=None)
def requests_get(url: str) -> requests.Response:
    time.sleep(DELAY)
    return requests.get(url, timeout=10)


def requests_pokeapi(url: str) -> dict:
    """GET with basic error handling. Cached by URL — repeated calls are free."""
    resp = requests_get(url)
    if not resp.ok:
        print(
            f"Error fetching from api, HTTP Code: {resp.status_code}. {resp.raw}"
        )
        return {}

    return resp.json()


def open_json_file(filename: str | Path) -> Any:
    with open(filename, "r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
# examples of how to read a byte->base64 encoded str back to python bytes
# the file is generated by another script
import base64

data = open_json_file("compiled_gen1_data.json")
# image = base64.b64decode(data[0]["front_sprite"])
[
    (
        base64.b64decode(data[i]["front_sprite"]),
        mo.image(base64.b64decode(data[i]["front_sprite"])),
        base64.b64decode(data[i]["back_sprite"]),
        mo.image(base64.b64decode(data[i]["back_sprite"])),
    )
    for i in range(0, POKE_ID_MAX)
]

## The next several cells will hit make network requests

Click the run button below to 'fire' them off.

In [ ]:
# marimo run button, good to block off expensive cells
poke_run_btn = mo.ui.run_button()
poke_run_btn

In [ ]:
mo.stop(
    not poke_run_btn.value,
    mo.md("☝️ Click button for 'expensive' api calls"),
)

# get move data for bulbasaur
# move_data = requests_pokeapi("https://pokeapi.co/api/v2/move/1")
# move_data

# get poke-api data for bulbasaur, all generations
poke_data = requests_pokeapi("https://pokeapi.co/api/v2/pokemon/1")
poke_data

In [ ]:
mo.stop(not poke_run_btn.value, "idk? my bff j?")
poke_sprites = {
    i: (
        mo.image(
            requests_get(
                f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/versions/generation-i/red-blue/transparent/{i}.png"
            ).content
        ),
        mo.image(
            requests_get(
                f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/versions/generation-i/red-blue/transparent/back/{i}.png"
            ).content
        ),
    )
    for i in range(1, 7)
}
# for i in range(1, 6):
#     png_resp = requests_get(
#         f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/versions/generation-i/red-blue/transparent/{i}.png"
#         # f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/versions/generation-i/red-blue/transparent/back/{i}.png"
#     )
#     if png_resp.headers["content-type"].startswith("image"):
#         poke_sprites[i] = png_resp.content
#     else:
#         print("http response was not a png image")
# [mo.image(src=poke_sprites[i]) for i in poke_sprites.keys()]
poke_sprites